# 可以考虑更新的方向

* 宏观序列是否考虑过去（比如用 LSTM）

## 本次修改：固定时点的因果宏观标准化

宏观变量现在按“观测年份”预先标准化，而不是在每个 Train/Val/Test 窗口内重新拟合：

* 年份 `y` 的所有月份只使用 `1990` 至 `y-1` 年的数据计算历史均值和标准差。
* 例如，2005 年无论处于测试集还是之后进入训练集，都始终使用 1990–2004 年的参数。
* 2006 年自身使用 1990–2005 年的参数，之后进入训练集时同样保持不变。
* 标准化后的宏观特征只计算一次，窗口循环只负责切片，从机制上避免同一记录跨窗口发生数值变化。

In [8]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

env= 'local'
# env= 'colab'
# env= 'kaggle'

# 核心参数
run_test = "quick" # "quick" 表示快速测试模式，使用更少的数据和更少的训练轮数；"full" 表示正式训练模式
random_seed = 42

best_k = 3

initial_train_months=180
val_months=12
test_months=12

epochs=100
batch_size=1024 # 1024合适吗?
lr=1e-3 # lr合适吗?
patience=25
dropout_rate = 0.1 # dropout率，防止过拟合


micro_normalize = 'normal' # 'normal' 或 'rank'

# 宏观变量固定时点标准化参数：年份 y 仅使用 [1990, y-1] 的月度观测
macro_norm_history_start_year = 1990
macro_norm_min_history_months = 12
macro_norm_eps = 1e-8
GLOBAL_DIST_TYPE = 't' # 't' 表示模型使用 t 分布, 'normal' 表示模型使用正态分布
epochs_type = 'fixed' # 'fixed' 或 'early_stopping'
windows_type = 'rolling' # 'expanding' 或 'rolling'
weights_type = 'uniform' # 'uniform' 或 'time_decay'

In [9]:

if env == 'kaggle':
    # This Python 3 environment comes with many helpful analytics libraries installed
    # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
    # For example, here's several helpful packages to load

    # Input data files are available in the read-only "../input/" directory
    # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    micro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/micro_data.parquet')
    macro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/MacroFactor_McCracken.parquet')
    # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
    # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

elif env == 'colab':
    # Colab 版本的代码
    # 1. 挂载 Google Drive，并从 Colab Secrets 读取 Kaggle 凭据
    # 请在 Colab 左侧 Secrets 中配置 KAGGLE_USERNAME 和 KAGGLE_KEY；不要把密钥写入 Notebook。
    from google.colab import drive, userdata
    drive.mount('/content/drive')

    for secret_name in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try:
            secret_value = userdata.get(secret_name)
        except Exception as exc:
            raise RuntimeError(
                f"请先在 Colab Secrets 中配置 {secret_name}，并允许此 Notebook 访问。"
            ) from exc
        if not secret_value:
            raise RuntimeError(f"Colab Secret {secret_name} 为空。")
        os.environ[secret_name] = secret_value

    # 2. 安装 kaggle 库 (如果尚未安装)
    !pip install -q kaggle

    # 3. 创建目录并下载数据集
    # -p 参数指定下载目录，--unzip 参数会自动解压
    !mkdir -p /content/mdndata
    !kaggle datasets download -d jianbinchenuc/mdndata-1990 -p /content/mdndata --unzip

    # 4. 验证文件是否已存在
    print("已下载的文件：", os.listdir('/content/mdndata'))
    micro_df = pd.read_parquet('/content/mdndata/micro_data.parquet') # 可以根据需要切换到 Colab 的路径
    macro_df = pd.read_parquet('/content/mdndata/MacroFactor_McCracken.parquet') # 可以根据需要切换到 Colab 的路径

elif env == 'local':
    micro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/micro_data.parquet') # local 路径
    macro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/MacroFactor_McCracken.parquet') # local 路径

else:
    raise ValueError("Invalid environment specified. Please choose 'kaggle', 'colab', or 'local'.")

In [10]:
print("Micro DataFrame shape:", micro_df['mthcaldt'].unique().shape)
print("Macro DataFrame shape:", macro_df.shape)

Micro DataFrame shape: (432,)
Macro DataFrame shape: (432, 20)


In [11]:
macro_df.tail(5)

,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
427,2024-08-31,0.000718,0.2,0.006105,-0.006401,0.003767,0.094333,0.00,7.121252,-0.002100,-0.022618,-0.006629,0.00,0.03,-0.1,0.002110,14.4084,0.51,-0.02,0.08
428,2024-09-30,0.000895,-0.1,-0.030903,0.008110,0.001463,0.000459,-0.01,7.212294,0.003887,0.008247,0.003711,0.00,-0.19,0.4,0.000322,19.6750,0.27,-0.23,0.14
429,2024-10-31,0.001598,-0.1,-0.029656,-0.002832,0.003543,-0.007867,0.01,7.210818,-0.001050,-0.023108,0.002598,-0.20,-0.26,1.6,-0.000073,17.6597,0.29,0.18,-0.03
430,2024-11-30,0.000075,0.0,0.055299,-0.002629,0.001225,0.002425,0.00,7.178545,0.000319,0.000541,-0.000157,-0.30,-0.24,0.3,0.000641,19.9478,0.80,0.59,-0.17
431,2024-12-31,0.001426,0.1,-0.078988,-0.001472,0.002754,-0.010575,0.00,7.161622,0.002139,0.025042,-0.002535,-0.19,-0.09,0.8,0.000686,15.9822,1.13,0.35,-0.11


In [12]:
micro_df.tail(5)

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,fcf_ocf,de_ratio,intcov_ratio,cash_ratio,curr_ratio,inv_turn,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,divyield,ffi49,linkdt,linkenddt
2226539,93436,2025-08-31,0.083044,333.87,1.076881e+09,1.598393e+09,3225449.0,3711.0,184996,2025-08-31,0.075492,88.014082,199.922156,11.609112,68.277633,192.695467,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.354329,0.670667,15.830137,1.224750,1.974345,5.164913,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,NaN,23.0,2010-06-29,2099-12-31
2226540,93436,2025-09-30,0.332015,444.72,1.478249e+09,1.966931e+09,3324000.0,3711.0,184996,2025-09-30,0.075492,88.014082,266.299401,15.943154,93.767794,264.634672,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.354329,0.670667,15.830137,1.224750,1.974345,5.164913,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,NaN,23.0,2010-06-29,2099-12-31
2226541,93436,2025-10-31,0.026624,456.56,1.518436e+09,2.024342e+09,3325819.0,3711.0,184996,2025-10-31,0.075492,88.014082,273.389222,16.367617,96.264221,271.680172,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.354329,0.670667,15.830137,1.224750,1.974345,5.164913,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,NaN,23.0,2010-06-29,2099-12-31
2226542,93436,2025-11-30,-0.057802,430.17,1.430668e+09,1.609832e+09,3325819.0,3711.0,184996,2025-11-30,0.054098,107.865207,298.729167,14.951796,90.797884,209.231062,0.053141,0.050903,0.232817,0.118495,0.068929,0.064976,0.174805,0.105348,1.234543,0.225793,0.525494,0.433960,0.661299,13.948424,1.285071,2.031625,5.582606,0.750824,22.058125,5.472834,1.254820,0.061736,0.0,0.0,-0.084740,NaN,23.0,2010-06-29,2099-12-31
2226543,93436,2025-12-31,0.045447,449.72,1.495687e+09,1.618322e+09,3325819.0,3711.0,184996,2025-12-31,0.054098,107.865207,312.305556,17.639306,107.118346,246.839292,0.053141,0.050903,0.232817,0.118495,0.068929,0.064976,0.174805,0.105348,1.234543,0.225793,0.525494,0.433960,0.661299,13.948424,1.285071,2.031625,5.582606,0.750824,22.058125,5.472834,1.254820,0.061736,0.0,0.0,-0.084740,NaN,23.0,2010-06-29,2099-12-31


In [13]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

# warnings.filterwarnings('ignore')

# ==========================================
# 全局随机种子锁死函数
# ==========================================
def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [14]:
micro_df.columns

Index(['permno', 'mthcaldt', 'mthret', 'mthprc', 'mthcap', 'mthvol', 'shrout',
       'siccd', 'gvkey', 'public_date', 'bm', 'evm', 'pe_exi', 'ps', 'pcf',
       'pfcf', 'npm', 'opmad', 'gpm', 'cfm', 'roe', 'aftret_invcapx', 'gprof',
       'debt_at', 'debt_ebitda', 'short_debt', 'ocf_lct', 'fcf_ocf',
       'de_ratio', 'intcov_ratio', 'cash_ratio', 'curr_ratio', 'inv_turn',
       'at_turn', 'rect_turn', 'pay_turn', 'sale_equity', 'rd_sale',
       'adv_sale', 'staff_sale', 'accrual', 'divyield', 'ffi49', 'linkdt',
       'linkenddt'],
      dtype='object')

In [15]:
# change the column names of macro_df to match the expected format
macro_df = macro_df.rename(columns={'mthcaldt':'date'})

# ==========================================
# 1. 外置数据预处理模块 (ETL & Feature Engineering)
# ==========================================
print(">>> [1/3] Loading and Preprocessing Data (Externalized)...")

# micro_cols 是微观特征列，macro_cols 是宏观特征列，后续我们会对它们分别进行不同的预处理
micro_cols=micro_df.columns.drop(['permno','mthcaldt', 'mthprc', 'shrout','siccd','gvkey', 'public_date','ffi49','linkdt','linkenddt']).tolist()
macro_cols=macro_df.columns.drop(['date']).tolist()

# 1.1 日期对齐与合并
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df['date'] = pd.to_datetime(macro_df['date']) + pd.offsets.MonthEnd(0)

df = pd.merge(micro_df, macro_df, left_on='mthcaldt', right_on='date', how='left')
df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
print(f"Data merged. micro_df shape: {micro_df.shape}, macro_df shape: {macro_df.shape}, Final shape: {df.shape}")
df.head(5)

>>> [1/3] Loading and Preprocessing Data (Externalized)...
Data merged. micro_df shape: (2226544, 45), macro_df shape: (432, 20), Final shape: (2226544, 65)


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,fcf_ocf,de_ratio,intcov_ratio,cash_ratio,curr_ratio,inv_turn,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,divyield,ffi49,linkdt,linkenddt,date,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,10001,1990-01-31,-0.018519,9.9375,10156.13,35255.0,1022.0,4920.0,012994,1990-01-31,0.917331,4.853678,2.743056,0.421441,7.769246,7.599586,0.050236,0.103437,0.134464,0.081263,0.151031,0.139773,0.164538,0.426239,2.590528,0.107954,0.339098,0.749436,2.235379,3.131479,0.271814,1.173138,91.308370,1.223659,7.855339,10.165835,3.959000,0.0,0.0,0.0,-0.005309,0.050633,31.0,1986-01-09,2017-08-31,1990-01-31,0.001114,0.0,0.072861,0.004778,0.000559,0.021997,-0.01,7.131699,-0.000003,0.031115,-0.003739,-0.10,-0.06,-0.1,0.000781,18.2350,1.37,0.03,0.04
1,10001,1990-02-28,-0.006289,9.8750,10092.25,14862.0,1022.0,4920.0,012994,1990-02-28,0.849363,4.626443,2.544529,0.403697,5.646409,7.695783,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,0.749436,2.084969,3.119863,0.273871,1.243238,96.509413,1.301493,8.324893,10.832774,4.015067,0.0,0.0,0.0,-0.024148,0.050000,31.0,1986-01-09,2017-08-31,1990-02-28,0.002334,0.0,-0.016349,-0.004656,0.008271,-0.079803,-0.01,7.346655,-0.001609,0.000240,0.002479,-0.22,-0.19,0.2,0.005485,23.4990,1.71,0.36,-0.25
2,10001,1990-03-31,0.012821,9.8750,10141.63,12727.0,1027.0,4920.0,012994,1990-03-31,0.849363,4.626443,2.576336,0.410743,5.744959,7.830102,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,0.749436,2.084969,3.119863,0.273871,1.243238,96.509413,1.301493,8.324893,10.832774,4.015067,0.0,0.0,0.0,-0.024148,0.049383,31.0,1986-01-09,2017-08-31,1990-03-31,0.002905,-0.1,-0.013831,0.005150,0.004845,0.026000,0.00,7.270313,-0.000426,-0.053552,0.005155,0.01,0.04,-0.2,-0.005538,23.3268,1.90,0.16,-0.06
3,10001,1990-04-30,0.000000,9.8750,10141.63,16645.0,1027.0,4920.0,012994,1990-04-30,0.849363,4.626443,2.544529,0.405672,5.674033,7.733434,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,0.749436,2.084969,3.119863,0.273871,1.243238,96.509413,1.301493,8.324893,10.832774,4.015067,0.0,0.0,0.0,-0.024148,0.050000,31.0,1986-01-09,2017-08-31,1990-04-30,0.001811,-0.1,-0.039778,0.004813,-0.005299,0.060018,-0.02,7.161622,-0.000234,-0.010451,-0.000162,0.04,0.14,0.1,-0.000015,18.8313,1.93,-0.04,-0.05
4,10001,1990-05-31,-0.012658,9.7500,10013.25,27885.0,1027.0,4920.0,012994,1990-05-31,0.911008,4.819588,2.480916,0.422964,4.424768,7.540098,0.056644,0.116415,0.148982,0.089212,0.158487,0.139023,0.182316,0.403169,2.211369,0.066927,0.693056,0.749436,1.943177,3.227166,0.263992,1.278309,89.841695,1.223747,8.168377,10.663666,3.601704,0.0,0.0,0.0,-0.047016,0.051282,31.0,1986-01-09,2017-08-31,1990-05-31,0.000347,0.2,0.031386,-0.006045,0.003234,-0.055663,-0.01,7.129298,0.000392,0.040279,0.000771,-0.02,0.02,0.1,-0.000790,20.4575,2.04,0.33,-0.11


In [16]:
df.tail(5)

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,fcf_ocf,de_ratio,intcov_ratio,cash_ratio,curr_ratio,inv_turn,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,divyield,ffi49,linkdt,linkenddt,date,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
2226539,93436,2025-08-31,0.083044,333.87,1.076881e+09,1.598393e+09,3225449.0,3711.0,184996,2025-08-31,0.075492,88.014082,199.922156,11.609112,68.277633,192.695467,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.354329,0.670667,15.830137,1.224750,1.974345,5.164913,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,NaN,23.0,2010-06-29,2099-12-31,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2226540,93436,2025-09-30,0.332015,444.72,1.478249e+09,1.966931e+09,3324000.0,3711.0,184996,2025-09-30,0.075492,88.014082,266.299401,15.943154,93.767794,264.634672,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.354329,0.670667,15.830137,1.224750,1.974345,5.164913,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,NaN,23.0,2010-06-29,2099-12-31,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2226541,93436,2025-10-31,0.026624,456.56,1.518436e+09,2.024342e+09,3325819.0,3711.0,184996,2025-10-31,0.075492,88.014082,273.389222,16.367617,96.264221,271.680172,0.063438,0.062317,0.236529,0.127793,0.082826,0.077265,0.177006,0.106271,1.144757,0.234716,0.529209,0.354329,0.670667,15.830137,1.224750,1.974345,5.164913,0.748345,23.193046,5.289922,1.258026,0.057302,0.0,0.0,-0.081217,NaN,23.0,2010-06-29,2099-12-31,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2226542,93436,2025-11-30,-0.057802,430.17,1.430668e+09,1.609832e+09,3325819.0,3711.0,184996,2025-11-30,0.054098,107.865207,298.729167,14.951796,90.797884,209.231062,0.053141,0.050903,0.232817,0.118495,0.068929,0.064976,0.174805,0.105348,1.234543,0.225793,0.525494,0.433960,0.661299,13.948424,1.285071,2.031625,5.582606,0.750824,22.058125,5.472834,1.254820,0.061736,0.0,0.0,-0.084740,NaN,23.0,2010-06-29,2099-12-31,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2226543,93436,2025-12-31,0.045447,449.72,1.495687e+09,1.618322e+09,3325819.0,3711.0,184996,2025-12-31,0.054098,107.865207,312.305556,17.639306,107.118346,246.839292,0.053141,0.050903,0.232817,0.118495,0.068929,0.064976,0.174805,0.105348,1.234543,0.225793,0.525494,0.433960,0.661299,13.948424,1.285071,2.031625,5.582606,0.750824,22.058125,5.472834,1.254820,0.061736,0.0,0.0,-0.084740,NaN,23.0,2010-06-29,2099-12-31,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# =========================================================
# 在无缝网格上安全计算特征
# 此时，1 行 (row) 绝对等于 1 个月 (month)
# =========================================================

# 1. 计算 t+1 收益率 (如果是断层前的最后一个月，它会极其安全地抓取到 NaN)
df['target_ret_final'] = df.groupby('permno')['mthret'].shift(-1)

# 2. 计算过去 12 个月动量 (跳过近1月，滚动11月)
df['log_ret'] = np.log1p(df['mthret']) # log1p means log(1+x)

# 注意：因为中间补了 NaN，所以遇到断层时，滚动的 11 个月里只要有效月份少于 min_periods (8)，
# 动量计算结果就会自动变成 NaN，完美避免了跨越几年的荒谬计算！
df['MOM12m'] = df.groupby('permno')['log_ret'].transform(
    lambda x: x.shift(1).rolling(window=11, min_periods=8).sum()
)
df['MOM12m'] = np.exp(df['MOM12m']) - 1
df = df.drop(columns=['log_ret'])  # 计算完动量后，log_ret 就没用了，可以删除了

print(" -> Reindexing and feature engineering complete!")
print(f" -> Total rows after reindexing: {len(df)}")
print(f" -> Total missing values in target_ret_final (should be > 0 due to gaps): {df['target_ret_final'].isna().sum()}")

# 3. 在完整的全局时间序列上，计算 12 个月滚动均值和标准差
df['mu_bench'] = df.groupby('permno')['mthret'].transform(lambda x: x.rolling(12, min_periods=8).mean())
df['sigma_bench'] = df.groupby('permno')['mthret'].transform(lambda x: x.rolling(12, min_periods=8).std())

# add new features to micro_cols
micro_cols = micro_cols+['MOM12m', 'mu_bench', 'sigma_bench']

/Users/jianbinchen/miniconda3/envs/research/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


 -> Reindexing and feature engineering complete!
 -> Total rows after reindexing: 2226544
 -> Total missing values in target_ret_final (should be > 0 due to gaps): 54028


In [18]:
# =========================================================
# 清洗没用的行
# =========================================================
# 每清洗一步都打印一下当前的行数和缺失值统计，确保我们没有过度清洗
print(">>> [2/3] Cleaning Data...")
# 把ffi49缺失的行变成0, 把这一列的类型改成整数（因为它是行业分类，缺失就当做0类）
df['ffi49'] = df['ffi49'].fillna(0).astype(int)
print(f"Initial rows: {len(df)}")
# 清洗target_ret_final缺失的行（这些行无法用于训练，因为没有标签了）
df = df.dropna(subset=['target_ret_final', 'public_date']).reset_index(drop=True)
# df = df[~df['target_ret_final'].isna()].reset_index(drop=True)
# 清洗public_date为空的行（这些行可能是数据错误，训练时可能会引入噪音）
# df = df[df['public_date'].notnull()].reset_index(drop=True)
# print(f"After dropping rows with missing public_date: {len(df)}")
print(f"After dropping rows with missing target_ret_final and public_date: {len(df)}")
# 清洗mthprc小于5的行（这些行可能是数据错误或者极端小盘股，训练时可能会引入噪音）
df = df[df['mthprc'] >= 5].reset_index(drop=True)
print(f"After dropping rows with mthprc < 5: {len(df)}")
# 清洗mthcaldt在指定范围外的行
df = df[ (df['mthcaldt'] >= pd.to_datetime('1991-01-31')) & (df['mthcaldt'] <= pd.to_datetime('2023-12-31')) ]  # 日期在1990-01-31到2023-12-31之间
print(f"After filtering by date: {len(df)}")

>>> [2/3] Cleaning Data...
Initial rows: 2226544
After dropping rows with missing target_ret_final and public_date: 2084584
After dropping rows with mthprc < 5: 1562411
After filtering by date: 1461459


In [19]:
# 1.3 缺失值预填充（微观变量）
# 先按“月份 + 行业”填充中位数；行业内仍为空时，再使用当月全市场中位数。
for feature in micro_cols:
    if feature in df.columns:
        df[feature] = df[feature].fillna(
            df.groupby(['mthcaldt', 'ffi49'])[feature].transform('median')
        )
        df[feature] = df[feature].fillna(
            df.groupby('mthcaldt')[feature].transform('median')
        )

# 1.4 微观特征：同月截面标准化。
# 这部分本来就只由该月截面决定，因此同一条记录跨窗口不会改变。
print(f" -> Applying {micro_normalize} normalization to Micro Features...")
micro_tensor_cols = []

if micro_normalize == 'rank':
    for col in micro_cols:
        norm_col = f'{col}_norm'
        df[norm_col] = df.groupby('mthcaldt')[col].transform(
            lambda x: (x.rank(pct=True) * 2) - 1
        ).fillna(0)
        micro_tensor_cols.append(norm_col)
elif micro_normalize == 'normal':
    for col in micro_cols:
        norm_col = f'{col}_norm'
        df[norm_col] = df.groupby('mthcaldt')[col].transform(
            lambda x: (x - x.mean()) / (x.std(ddof=1) + 1e-8)
        ).replace([np.inf, -np.inf], np.nan).fillna(0)
        micro_tensor_cols.append(norm_col)
else:
    raise ValueError("micro_normalize 只能是 'normal' 或 'rank'")


def make_causal_yearly_macro_zscores(
    macro_data,
    feature_cols,
    date_col='date',
    history_start_year=1990,
    min_history_months=12,
    eps=1e-8,
):
    """
    为每个宏观观测生成一次性、不可随训练窗口变化的因果 Z-score。

    对目标年份 y：
      mean_y, std_y 只由 history_start_year 至 y-1 年的月度数据计算；
      y 年所有月份统一使用 mean_y 和 std_y；
      以后 y 年进入训练集时，直接复用已经算好的数值，不重新拟合 scaler。

    当前年份的宏观值若缺失，用对应的历史均值填充，因此其 Z-score 为 0；
    历史不足或历史标准差为 0 时保留 NaN，并在下方显式报错，避免静默污染模型。
    """
    feature_cols = list(feature_cols)
    base = macro_data[[date_col] + feature_cols].copy()
    base[date_col] = pd.to_datetime(base[date_col]) + pd.offsets.MonthEnd(0)
    base = base.sort_values(date_col).reset_index(drop=True)

    duplicated_dates = base.loc[
        base[date_col].duplicated(keep=False), date_col
    ].drop_duplicates()
    if not duplicated_dates.empty:
        raise ValueError(
            "macro_df 必须每个月只有一行；发现重复月份："
            f"{duplicated_dates.head().dt.strftime('%Y-%m-%d').tolist()}"
        )

    # 明确从 1990 年开始累计，避免 macro_df 中更早年份意外进入统计量。
    base = base[base[date_col].dt.year >= history_start_year].reset_index(drop=True)
    years = base[date_col].dt.year
    norm_cols = [f'{col}_causal_z' for col in feature_cols]

    normalized = base[[date_col]].copy()
    normalized['macro_norm_ref_start_year'] = history_start_year
    normalized['macro_norm_ref_end_year'] = years - 1
    normalized[norm_cols] = np.nan

    parameter_frames = []

    for target_year in np.sort(years.unique()):
        history_mask = (years >= history_start_year) & (years < target_year)
        current_mask = years.eq(target_year)

        history = base.loc[history_mask, feature_cols]
        means = history.mean(axis=0)
        stds = history.std(axis=0, ddof=1) # 当你设置 ddof=1 时，计算标准差（std）的分母使用的是 $N - 1$（样本标准差），而不是 $N$（总体标准差）。
        counts = history.count()

        valid = (
            counts.ge(min_history_months)
            & means.notna()
            & stds.notna()
            & stds.abs().gt(eps)
        )

        # 当前年份缺失的宏观值只用历史均值填充，严格不使用未来信息。
        current = base.loc[current_mask, feature_cols].copy().fillna(means)
        safe_stds = stds.where(valid)
        z_values = current.sub(means, axis='columns').div(
            safe_stds, axis='columns'
        )
        z_values = z_values.replace([np.inf, -np.inf], np.nan)
        z_values.columns = norm_cols
        normalized.loc[current_mask, norm_cols] = z_values.to_numpy()

        history_dates = base.loc[history_mask, date_col]
        parameter_frames.append(pd.DataFrame({
            'target_year': target_year,
            'feature': feature_cols,
            'history_start_date': (
                history_dates.min() if not history_dates.empty else pd.NaT
            ),
            'history_end_date': (
                history_dates.max() if not history_dates.empty else pd.NaT
            ),
            'n_obs': counts.reindex(feature_cols).to_numpy(),
            'mean': means.reindex(feature_cols).to_numpy(),
            'std': stds.reindex(feature_cols).to_numpy(),
            'is_valid': valid.reindex(feature_cols).to_numpy(),
        }))

    normalized[norm_cols] = normalized[norm_cols].astype(np.float32)
    macro_norm_params = pd.concat(parameter_frames, ignore_index=True)
    return normalized, norm_cols, macro_norm_params


# 1.5 宏观特征：按年份做严格滞后一期的 expanding Z-score。
# 关键点：它在进入窗口循环前只计算一次，之后任何窗口都只做索引切片。
print(" -> Precomputing immutable causal yearly Z-scores for Macro Features...")
macro_norm_df, macro_tensor_cols, macro_norm_params = make_causal_yearly_macro_zscores(
    macro_data=macro_df,
    feature_cols=macro_cols,
    date_col='date',
    history_start_year=macro_norm_history_start_year,
    min_history_months=macro_norm_min_history_months,
    eps=macro_norm_eps,
)

# 允许重复运行本单元：先删除旧的标准化列和审计列，再重新合并。
macro_audit_cols = ['macro_norm_ref_start_year', 'macro_norm_ref_end_year']
df = df.drop(columns=macro_tensor_cols + macro_audit_cols, errors='ignore')
df = df.merge(
    macro_norm_df[['date'] + macro_audit_cols + macro_tensor_cols],
    on='date',
    how='left',
    validate='many_to_one',
)

# 对进入模型区间的数据做强校验，禁止 NaN/Inf 静默进入神经网络。
macro_values = df[macro_tensor_cols].to_numpy(dtype=np.float64, copy=False)
if not np.isfinite(macro_values).all():
    bad_counts = (
        df[macro_tensor_cols]
        .replace([np.inf, -np.inf], np.nan)
        .isna()
        .sum()
    )
    bad_counts = bad_counts[bad_counts.gt(0)].sort_values(ascending=False)
    first_bad_date = df.loc[
        ~np.isfinite(macro_values).all(axis=1), 'mthcaldt'
    ].min()
    raise ValueError(
        "宏观因果标准化后仍存在非有限值。"
        f"最早问题日期：{first_bad_date}；问题列：{bad_counts.head().to_dict()}。"
        "请检查 1990 年历史是否完整，或调低 macro_norm_min_history_months。"
    )

# 审计示例：2005 年必须且只允许引用 1990-2004。
audit_year = 2005
if df['mthcaldt'].dt.year.eq(audit_year).any():
    audit_refs = df.loc[
        df['mthcaldt'].dt.year.eq(audit_year),
        macro_audit_cols,
    ].drop_duplicates()
    if len(audit_refs) != 1:
        raise AssertionError(f"{audit_year} 年出现了不一致的宏观标准化参考区间。")

    ref_start = int(audit_refs.iloc[0]['macro_norm_ref_start_year'])
    ref_end = int(audit_refs.iloc[0]['macro_norm_ref_end_year'])
    if ref_start != macro_norm_history_start_year or ref_end != audit_year - 1:
        raise AssertionError(
            f"{audit_year} 年参考区间错误：得到 {ref_start}-{ref_end}，"
            f"预期 {macro_norm_history_start_year}-{audit_year - 1}。"
        )

    audit_params = macro_norm_params.loc[
        macro_norm_params['target_year'].eq(audit_year)
    ]
    print(
        f"    Audit {audit_year}: fixed reference = {ref_start}-{ref_end}; "
        f"history dates = {audit_params['history_start_date'].min():%Y-%m-%d} "
        f"to {audit_params['history_end_date'].max():%Y-%m-%d}."
    )

# 1.6 固化数据集
print(" -> Finalizing dataset...")
print(f"Initial shape before finalization: {df.shape}")
processed_df = df.copy()
print(f"Data preprocessing complete! Final shape: {processed_df.shape}")
print(f"Micro Features: {len(micro_tensor_cols)}, Macro Features: {len(macro_tensor_cols)}")

if env == 'local':
    processed_df.to_parquet('../data/processed_df.parquet', index=False)

pd.concat([processed_df.iloc[:5], processed_df.iloc[-5:]])

 -> Applying normal normalization to Micro Features...
 -> Precomputing immutable causal yearly Z-scores for Macro Features...
    Audit 2005: fixed reference = 1990-2004; history dates = 1990-01-31 to 2004-12-31.
 -> Finalizing dataset...
Initial shape before finalization: (1461459, 128)
Data preprocessing complete! Final shape: (1461459, 128)
Micro Features: 38, Macro Features: 19


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,fcf_ocf,de_ratio,intcov_ratio,cash_ratio,curr_ratio,inv_turn,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,divyield,ffi49,linkdt,linkenddt,date,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD,target_ret_final,MOM12m,mu_bench,sigma_bench,mthret_norm,mthcap_norm,mthvol_norm,bm_norm,evm_norm,pe_exi_norm,ps_norm,pcf_norm,pfcf_norm,npm_norm,opmad_norm,gpm_norm,cfm_norm,roe_norm,aftret_invcapx_norm,gprof_norm,debt_at_norm,debt_ebitda_norm,short_debt_norm,ocf_lct_norm,fcf_ocf_norm,de_ratio_norm,intcov_ratio_norm,cash_ratio_norm,curr_ratio_norm,inv_turn_norm,at_turn_norm,rect_turn_norm,pay_turn_norm,sale_equity_norm,rd_sale_norm,adv_sale_norm,staff_sale_norm,accrual_norm,divyield_norm,MOM12m_norm,mu_bench_norm,sigma_bench_norm,macro_norm_ref_start_year,macro_norm_ref_end_year,PAYEMS_causal_z,UNRATE_causal_z,CLAIMSx_causal_z,INDPRO_causal_z,DPCERA3M086SBEA_causal_z,AMDMNOx_causal_z,ISRATIOx_causal_z,HOUST_causal_z,M2SL_causal_z,NONBORRES_causal_z,BUSLOANS_causal_z,FEDFUNDS_causal_z,CP3Mx_causal_z,UEMPMEAN_causal_z,CPIAUCSL_causal_z,VIXCLSx_causal_z,BAAFFM_causal_z,TERM_SPREAD_causal_z,CREDIT_SPREAD_causal_z
0,10001,1991-01-31,0.013158,9.625,1.014475e+04,3.839800e+04,1054.0,4920.0,012994,1991-01-31,0.896385,5.690808,3.307560,0.445982,4.872598,14.070388,0.044401,0.099749,0.138304,0.082956,0.112235,0.112708,0.161306,0.392076,2.430626,0.061202,0.672101,0.380676,1.796365,2.770452,0.316439,1.340247,87.798432,1.166318,8.789413,10.931698,3.261452,0.000000,0.0,0.0,-0.055775,0.057143,31,1986-01-09,2017-08-31,1991-01-31,-0.001000,0.1,0.023836,-0.005920,-0.000778,0.040338,0.02,6.876265,0.001402,-0.048901,0.000930,-0.50,-0.11,0.1,0.001485,22.5845,3.12,0.01,0.12,0.012987,0.009425,0.002110,0.022682,-0.504652,-0.266440,-0.417250,-0.114176,-0.008713,-0.318060,-0.263231,-0.076235,0.101985,0.023106,0.022281,0.000860,0.021162,-0.022824,-0.008157,-0.540026,0.819039,-0.033047,-0.905843,0.102895,0.019306,-2.479270e-01,-0.079284,-0.193195,-0.141815,0.812674,0.111213,-0.071472,0.026097,-0.008447,-0.165090,-0.259057,-0.530427,-0.277413,0.822701,0.061209,-0.153869,-1.672020,1990,1990,-0.770984,0.199637,0.008891,-0.919695,-0.487627,0.988399,1.456512,-1.986470,0.688952,-1.899404,0.161413,-3.676451,-0.746807,0.221522,0.463033,0.003050,2.541223,-0.386150,0.658339
1,10001,1991-02-28,0.012987,9.750,1.027650e+04,2.452500e+04,1054.0,4920.0,012994,1991-02-28,0.975332,5.299653,3.037383,0.434947,5.205927,14.253121,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,0.380676,1.755852,3.063990,0.289359,1.348320,89.490628,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,0.056410,31,1986-01-09,2017-08-31,1991-02-28,-0.001892,0.1,-0.041536,-0.004931,-0.003955,-0.029746,0.02,6.682109,0.001999,-0.142799,-0.007471,-0.40,-0.70,-0.3,-0.000014,26.9318,3.54,0.53,0.01,-0.012244,0.029179,0.003717,0.022716,-0.684256,-0.260183,-0.452887,-0.018401,0.013910,-0.377271,-0.230634,-0.024408,0.120973,0.050088,0.048682,-0.028918,0.044472,-0.023129,0.038542,-0.507792,0.749763,0.003524,-0.905279,0.067154,0.019065,-8.853422e-02,-0.063137,-0.155730,-0.091358,0.551104,0.137736,-0.075506,0.013081,-0.013282,-0.055696,-0.366452,-0.517821,-0.112377,0.699829,0.016855,-0.281518,-1.688633,1990,1990,-1.267799,0.199637,-1.657304,-0.745792,-1.308811,-0.452644,1.456512,-3.666820,0.918470,-5.401950,-1.821742,-2.837717,-6.755208,-1.905091,0.019033,0.962087,3.597970,1.953243,-0.097320
2,10001,1991-03-31,-0.012244,9.500,1.001300e+04,1.243900e+04,1054.0,4920.0,012994,1991-03-31,0.975332,5.299653,2.959502,0.423795,5.072442,13.887656,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,0.

In [20]:
# print any row with na in processed_df
processed_df[processed_df.isna().any(axis=1)]

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,fcf_ocf,de_ratio,intcov_ratio,cash_ratio,curr_ratio,inv_turn,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,divyield,ffi49,linkdt,linkenddt,date,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD,target_ret_final,MOM12m,mu_bench,sigma_bench,mthret_norm,mthcap_norm,mthvol_norm,bm_norm,evm_norm,pe_exi_norm,ps_norm,pcf_norm,pfcf_norm,npm_norm,opmad_norm,gpm_norm,cfm_norm,roe_norm,aftret_invcapx_norm,gprof_norm,debt_at_norm,debt_ebitda_norm,short_debt_norm,ocf_lct_norm,fcf_ocf_norm,de_ratio_norm,intcov_ratio_norm,cash_ratio_norm,curr_ratio_norm,inv_turn_norm,at_turn_norm,rect_turn_norm,pay_turn_norm,sale_equity_norm,rd_sale_norm,adv_sale_norm,staff_sale_norm,accrual_norm,divyield_norm,MOM12m_norm,mu_bench_norm,sigma_bench_norm


In [ ]:
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col
        
    def expanding_window_generator(self, initial_train_months=180, val_months=12, test_months=12):
        dates = np.sort(self.df[self.date_col].unique())
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months + test_months <= len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months
            
    # （固定 15 年滚动窗口版）
    def rolling_window_generator(self, train_months=180, val_months=12, test_months=12):
        """
        固定训练窗口长度的滚动窗口生成器。
        参数：
        train_months: 固定训练集包含的月数。默认 180 个月(15年)。
        """
        # 确保日期是有序的
        dates = np.sort(self.df[self.date_col].unique())
        
        # 初始时，当前切分点就是训练集的总长度（180个月）
        current_split_idx = train_months
        
        # 只要后面还有足够的月份可以分给验证集和测试集，就继续滚动
        while current_split_idx + val_months + test_months <= len(dates):
            
            # 【核心修改点】让训练集的起点跟着一起滚动，确保长度永远等于 train_months
            start_idx = current_split_idx - train_months
            
            # 切分对应的日期数组
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            # 根据日期过滤出对应的数据集
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            
            # 向前推進一个测试窗口的长度
            current_split_idx += test_months

In [ ]:
# ==========================================
# 2. 模型定义：结构化 MDN (Mixture Density Network) + FiLM 调制
# ==========================================
class StructuredMDN_with_FiLM(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6, 
                 micro_hidden_dim=64, macro_hidden_dim=32, num_components=3, 
                 dropout_rate=0.1, dist_type='normal'):
        """
        结构化 MDN (加入 FiLM 调制机制)
        - 保持 2D 截面输入，不引入序列记忆 (无 GRU)
        - 宏观特征不仅预测 Regime 概率，还直接调控微观特征的表达
        """
        super(StructuredMDN_with_FiLM, self).__init__()
        self.num_components = num_components
        self.dist_type = dist_type

        # ++++++++++++++++++++++++++++++++++++++++++
        # 新增：行业实体嵌入层 (Entity Embedding)
        # ++++++++++++++++++++++++++++++++++++++++++
        self.ind_embedding = nn.Embedding(num_embeddings=num_industries, embedding_dim=ind_emb_dim)
        
        # 计算拼接后的微观特征总维度 = 连续特征维度 + 行业嵌入维度(6)
        total_micro_dim = micro_dim + ind_emb_dim
        # ==========================================
        # 通道 A: Macro Network (标准 MLP)
        # 输入: (Batch, Macro_Dim) 截面宏观数据
        # ==========================================
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(macro_hidden_dim , macro_hidden_dim//2), # 🌟 优化：增加一层，让宏观特征更丰富
            nn.LayerNorm(macro_hidden_dim//2),
            nn.ELU()
        )
        
        # 门控预测头：基于宏观隐状态预测机制概率 pi
        self.pi_head = nn.Linear(macro_hidden_dim//2, num_components)
        
        # ==========================================
        # ★ FiLM 调制发生器 (Modulation Generators) ★
        # ==========================================
        # 目标是对齐微观网络的最后一层输出维度
        target_dim = 8
        
        # Gamma 负责“缩放(Scale)”微观特征，Beta 负责“平移(Shift)”微观特征
        self.film_gamma = nn.Linear(macro_hidden_dim//2, target_dim)
        self.film_beta = nn.Linear(macro_hidden_dim//2, target_dim)
        
        # ==========================================
        # 通道 B: Micro Network (专家网络)
        # 输入: (Batch, Micro_Dim) 截面微观数据
        # ==========================================
        self.micro_extractor = nn.Sequential(
            nn.Linear(total_micro_dim, micro_hidden_dim),
            nn.LayerNorm(micro_hidden_dim), # 使用 LayerNorm 防止截面前视偏差
            nn.ELU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.LayerNorm(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(micro_hidden_dim // 2, target_dim),
            nn.LayerNorm(target_dim),
            nn.ELU()
        )
        
        # 预测头 (Heads): 注意，这里输入的是被 FiLM 调制后的特征
        self.mu_head = nn.Linear(target_dim, num_components)
        self.sigma_head = nn.Linear(target_dim, num_components)
        # 🌟 仅在选择了 t 分布时，才挂载自由度预测头
        if self.dist_type == 't':
            self.nu_head = nn.Linear(target_dim, num_components)    

        # ==========================================
        # 🌟 优化：MDN 安全初始化策略
        # ==========================================
        # 1. 均值头：初始化为极小的值，防止初始预期收益率过高
        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        
        # 2. Sigma 头：让 softplus 初始化的结果在 1.0 左右
        # softplus(0.54) 大约等于 1.0
        # nn.init.constant_(self.sigma_head.bias, 0.5413)
        nn.init.constant_(self.sigma_head.bias, -2.35)
        
        if self.dist_type == 't':
            # 自由度头：初始化 nu 较大（例如 10），使其初始状态接近正态分布
            # 因为你的代码里有 + 2.01 的截断，softplus(x) + 2.01 = 10 -> x ≈ 2.07
            # nn.init.constant_(self.nu_head.bias, 2.07)  
            nn.init.constant_(self.nu_head.bias, 5)

    def forward(self, x_macro, x_micro, x_micro_ind):
        """
        x_macro: (Batch, Macro_Dim) - 当前时刻的宏观因子 (2D张量)
        x_micro: (Batch, Micro_Dim) - 当前时刻的微观因子 (2D张量)
        x_micro_ind: (Batch,) - 行业整数索引 0~47 (Long Tensor)
        """
        # ------------------------------------------
        # 1. 宏观特征提取
        # ------------------------------------------
        h_macro = self.macro_net(x_macro) # 形状: (Batch, macro_hidden_dim)
        
        # 预测门控权重
        pi_logits = self.pi_head(h_macro)
        
        # ------------------------------------------
        # 2. 生成 FiLM 参数 (根据当前宏观环境)
        # ------------------------------------------
        gamma = self.film_gamma(h_macro) # 形状: (Batch, target_dim)
        beta = self.film_beta(h_macro)   # 形状: (Batch, target_dim)
        
        # ++++++++++++++++++++++++++++++++++++++++++
        # 3. 新增：Embedding 降维与特征拼接
        # ++++++++++++++++++++++++++++++++++++++++++
        # 将 [0, 1, ..., 47] 的整数索引，转化为 6 维稠密向量
        ind_emb = self.ind_embedding(x_micro_ind) # 形状: (Batch, 6)
        
        # 在特征维度 (dim=-1) 上拼接连续特征和行业 Embedding
        x_micro_combined = torch.cat([x_micro, ind_emb], dim=-1)
        
        # 修改：将拼接后的组合特征送入提取器
        h_micro = self.micro_extractor(x_micro_combined)
        
        # ------------------------------------------
        # 4. ★ 核心：执行 FiLM 调制 ★
        # ------------------------------------------
        # 宏观环境直接介入微观特征的表达。
        # (1.0 + gamma) 确保网络初始阶段 gamma 接近 0 时，维持原特征。
        h_modulated = (1.0 + gamma) * h_micro + beta
        
        # ------------------------------------------
        # 5. 专家网络最终预测
        # ------------------------------------------
        # mu = self.mu_head(h_modulated)
        # Mu 路径：直接使用纯粹的、未受宏观污染的微观特征
        mu = self.mu_head(h_modulated)
        sigma = nn.functional.softplus(self.sigma_head(h_modulated)) + 0.05
        
        # 3. 🌟 根据开关决定 Nu
        nu = None # 默认为空
        if self.dist_type == 't':
            # 保证自由度严格大于 2，以确保方差存在
            nu = nn.functional.softplus(self.nu_head(h_modulated)) + 2.01
            
        # 统一返回4个值，如果是 normal，nu 就是 None
        return pi_logits, mu, sigma, nu


In [ ]:

# ==========================================
# 4. 稳健的 NLL 损失函数 (支持时间衰减权重)
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, target, weights, dist_type='normal'):
    """
    weights: shape (batch_size, 1)，距离现在越近权重大
    """
    log_pi = torch.log_softmax(pi_logits, dim=-1)

# 🌟 核心开关：底层分布切换
    if dist_type == 't':
        dist = torch.distributions.StudentT(df=nu, loc=mu, scale=sigma)
    else:
        dist = torch.distributions.Normal(mu, sigma)
    
    # 分布的 log_prob
    log_dist = dist.log_prob(target.expand_as(mu))
    log_mix = log_pi + log_dist
    
    # 每个样本基础的 NLL loss
    loss = -torch.logsumexp(log_mix, dim=-1) 
    
    # 乘以时间衰减权重后求平均
    # 确保 weights 的 shape 与 loss 对齐
    weighted_loss = loss * weights.squeeze()
    
    return weighted_loss.sum()/ weights.sum()

In [12]:
# ==========================================
# 5. 极速版双通道训练循环 (All-in-VRAM 加速)
# ==========================================
def train_structured_mdn(model, 
                         X_train_macro, X_train_micro, X_train_industry, y_train, w_train, 
                         X_val_macro=None, X_val_micro=None, X_val_industry=None, y_val=None, w_val=None,
                         epochs=500, batch_size=1024, lr=1e-3, patience=25): # 注意 batch_size 变了！
    if run_test == "quick":
        epochs = 2
        batch_size = 512
        patience = 1

    # 2. 🌟 智能判定：只有当验证标签不为空时，才实例化验证集 Loader
    has_validation = (y_val is not None)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.8, patience=8, min_lr=1e-6,
    )
    # 极度清爽，直接将传入的 Tensor 打包，不需要任何格式转换和设备转移！
    tr_dataset = TensorDataset(X_train_macro, X_train_micro, X_train_industry, y_train.unsqueeze(1), w_train.unsqueeze(1))
    train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    train_loss_history = []

    # 3. 🌟 智能判定：只有当验证标签不为空时，才实例化验证集 Loader
    if has_validation:
        va_dataset = TensorDataset(X_val_macro, X_val_micro, X_val_industry, y_val.unsqueeze(1), w_val.unsqueeze(1))
        val_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)
        
        epochs_no_improve = 0
        best_model_state = None
    best_val_loss = float('inf')
    val_loss_history = []
    
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        # 注意看：这里去掉了 .to(device)！因为数据切片本来就在显存里了！
        for b_macro, b_micro, b_industry, b_y, b_w in train_loader:
            optimizer.zero_grad()
            
            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            loss.backward()
            # 梯度裁剪，防止爆炸(一定要吗?)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        train_loss_history.append(train_loss)
        
        # Validation
        if has_validation:
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for b_macro, b_micro, b_industry, b_y, b_w in val_loader:
                    pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
                    loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)
                    val_loss += loss.item() * b_macro.size(0)
            val_loss /= len(val_loader.dataset)
            val_loss_history.append(val_loss)

            scheduler.step(val_loss)
                
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    break
    if has_validation and best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, best_val_loss, train_loss_history, val_loss_history

In [13]:
# ==========================================
# 6. 极速版双通道训练循环 (All-in-VRAM 加速) 固定epochs版本, 第一期100个epochs, 后面都是50个. 
# ==========================================
def train_structured_mdn_fixed_epochs(model, 
                         X_train_macro, X_train_micro, X_train_industry, y_train, w_train, 
                         X_val_macro=None, X_val_micro=None, X_val_industry=None, y_val=None, w_val=None,
                         epochs=100, batch_size=1024, lr=1e-3, patience=25, is_first_window=False): # 注意 batch_size 变了！
    # if run_test == "quick":
    #     batch_size = 512

    # ==========================================
    # 永远合并 train + val，区别只有 epoch 数
    # 第一轮：100 epoch；后续轮：50 epoch
    # ==========================================
    epochs_to_run = (2 if run_test == "quick" else 100) if is_first_window else (1 if run_test == "quick" else 50)

    combined_dataset = TensorDataset(
        torch.cat([X_train_macro,    X_val_macro],    dim=0),
        torch.cat([X_train_micro,    X_val_micro],    dim=0),
        torch.cat([X_train_industry, X_val_industry], dim=0),
        torch.cat([y_train,          y_val],          dim=0).unsqueeze(1),
        torch.cat([w_train,          w_val],          dim=0).unsqueeze(1),
    )
    train_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=True)


    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.8, patience=8, min_lr=1e-6,
    )
    
    # 🌟 新增：用于记录 Loss 历史
    train_loss_history = []
    val_loss_history = []
    
    for epoch in range(epochs_to_run):
        model.train()
        train_loss = 0.0
        # 注意看：这里去掉了 .to(device)！因为数据切片本来就在显存里了！
        for b_macro, b_micro, b_industry, b_y, b_w in train_loader:
            optimizer.zero_grad()

            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        train_loss_history.append(train_loss)
        val_loss_history.append(train_loss)   # 无独立 val，用 train_loss 占位
        scheduler.step(train_loss)

    return model, train_loss, train_loss_history, val_loss_history

In [14]:
# ==========================================
# 基于半衰期的时间衰减权重计算函数
# ==========================================
def calculate_time_decay_weights(df_dates, half_life_months=36):
    """
    计算基于半衰期的指数衰减权重。
    参数：
    df_dates: pd.Series, 包含 mthcaldt 日期
    half_life_months: 半衰期(月)。默认 36 个月(3年)权重减半。
    """
    max_date = df_dates.max()
    # 计算每个样本距离该窗口最新日期的月数差
    months_diff = (max_date.year - df_dates.dt.year) * 12 + (max_date.month - df_dates.dt.month)
    
    # 衰减因子 lambda: (0.5)^(1/half_life)
    decay_factor = np.power(0.5, 1.0 / half_life_months)
    
    # 计算基础权重
    weights = np.power(decay_factor, months_diff).values
    
    # 【极度关键】归一化权重，使得权重的均值为 1。
    # 如果不归一化，整个 Loss 的绝对值会变小，导致梯度的步长变相缩小！
    weights = weights / weights.mean() 
    return weights

In [ ]:
if __name__ == "__main__":
    seed_everything(seed=random_seed)
    print()
    print(">>> [2/3] Initializing Data Window Generator and Global VRAM Tensors...")

    # 1. 实例化时间窗口生成器。索引必须与全局 Tensor 一一对应。
    processed_df = processed_df.reset_index(drop=True)
    generator = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    if windows_type == 'expanding':
        window_gen = generator.expanding_window_generator(
            initial_train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )
    else:
        window_gen = generator.rolling_window_generator(
            train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )

    # ==============================================================
    # 全局数据一次性放入设备。
    # macro_tensor_cols 已在预处理阶段完成“按年份、严格滞后、固定不变”的标准化。
    # 因此这里以及后续窗口内绝不能重新计算 mean/std/median/IQR。
    # ==============================================================
    print("    -> Pushing the entire dataset into device memory once...")

    if processed_df[macro_tensor_cols].isna().any().any():
        raise ValueError("macro_tensor_cols 中仍有 NaN，停止训练。")

    global_X_mac = torch.from_numpy(
        processed_df[macro_tensor_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)
    global_X_mic = torch.from_numpy(
        processed_df[micro_tensor_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)
    global_ind = torch.from_numpy(
        processed_df['ffi49'].to_numpy(dtype=np.int64, copy=True)
    ).to(device)
    global_Y = torch.from_numpy(
        processed_df['target_ret_final'].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    history_logs = []
    all_oos_predictions = []
    macro_dim = len(macro_tensor_cols)
    micro_dim = len(micro_tensor_cols)
    global_model = StructuredMDN_with_FiLM(
        macro_dim,
        micro_dim,
        num_industries=50,
        ind_emb_dim=6,
        num_components=best_k,
        dist_type=GLOBAL_DIST_TYPE,
        dropout_rate=dropout_rate,
    ).to(device)

    print()
    print(
        f">>> [3/3] Starting {windows_type.title()} Window Structured MDN "
        "Training (immutable causal normalization)..."
    )

    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print()
        print(
            f"[Window {window_idx + 1}] "
            f"Train: {info['train'][0]}~{info['train'][1]} | "
            f"Val: {info['val'][0]}~{info['val'][1]} | "
            f"Test: {info['test'][0]}~{info['test'][1]}"
        )
        start_time = time.time()

        # 提取各集合在全局大表中的行索引。
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 直接切片固定的宏观标准化值。
        # 同一条 2005 年记录在测试阶段和之后进入训练阶段时，数值完全相同。
        X_tr_mac = global_X_mac[idx_tr]
        X_va_mac = global_X_mac[idx_va]
        X_te_mac = global_X_mac[idx_te]

        X_tr_mic, X_tr_ind, Y_tr = (
            global_X_mic[idx_tr], global_ind[idx_tr], global_Y[idx_tr]
        )
        X_va_mic, X_va_ind, Y_va = (
            global_X_mic[idx_va], global_ind[idx_va], global_Y[idx_va]
        )
        X_te_mic, X_te_ind = global_X_mic[idx_te], global_ind[idx_te]

        # 动态样本权重。
        if weights_type == 'time_decay':
            if epochs_type == 'fixed':
                # fixed 模式实际把 train+val 一起训练；必须在联合时间轴上一次性算权重，
                # 否则分别归一化会抹掉两组之间的相对新旧关系。
                combined_dates = pd.concat(
                    [train_df['mthcaldt'], val_df['mthcaldt']],
                    ignore_index=True,
                )
                combined_weights = calculate_time_decay_weights(
                    combined_dates, half_life_months=36
                )
                w_train_np = combined_weights[:len(train_df)]
                w_val_np = combined_weights[len(train_df):]
            else:
                w_train_np = calculate_time_decay_weights(
                    train_df['mthcaldt'], half_life_months=36
                )
                w_val_np = calculate_time_decay_weights(
                    val_df['mthcaldt'], half_life_months=12
                )
        else:
            w_train_np = np.ones(len(train_df), dtype=np.float32)
            w_val_np = np.ones(len(val_df), dtype=np.float32)

        w_train = torch.tensor(w_train_np, dtype=torch.float32, device=device)
        w_val = torch.tensor(w_val_np, dtype=torch.float32, device=device)

        current_lr = 1e-3 if window_idx == 0 else 3e-4
        is_first_window = window_idx == 0

        if epochs_type == 'fixed':
            global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn_fixed_epochs(
                model=global_model,
                X_train_macro=X_tr_mac,
                X_train_micro=X_tr_mic,
                X_train_industry=X_tr_ind,
                y_train=Y_tr,
                w_train=w_train,
                X_val_macro=X_va_mac,
                X_val_micro=X_va_mic,
                X_val_industry=X_va_ind,
                y_val=Y_va,
                w_val=w_val,
                lr=current_lr,
                epochs=epochs,
                patience=patience,
                batch_size=batch_size,
                is_first_window=is_first_window,
            )
        else:
            global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn(
                model=global_model,
                X_train_macro=X_tr_mac,
                X_train_micro=X_tr_mic,
                X_train_industry=X_tr_ind,
                y_train=Y_tr,
                w_train=w_train,
                X_val_macro=X_va_mac,
                X_val_micro=X_va_mic,
                X_val_industry=X_va_ind,
                y_val=Y_va,
                w_val=w_val,
                lr=current_lr,
                epochs=epochs,
                patience=patience,
                batch_size=batch_size,
            )

            # 基于 Train + Val 联合集合做短暂 fine-tune。
            best_epoch = len(tr_hist)
            finetune_epochs = max(5, best_epoch // 10)
            print(
                f"  Phase 1 done: best_epoch={best_epoch}, "
                f"finetune_epochs={finetune_epochs}"
            )

            X_tr_mac_ft = torch.cat([X_tr_mac, X_va_mac], dim=0)
            X_tr_mic_ft = torch.cat([X_tr_mic, X_va_mic], dim=0)
            X_tr_ind_ft = torch.cat([X_tr_ind, X_va_ind], dim=0)
            Y_tr_ft = torch.cat([Y_tr, Y_va], dim=0)

            train_val_df = pd.concat([train_df, val_df], ignore_index=True)
            if weights_type == 'time_decay':
                w_ft_np = calculate_time_decay_weights(
                    train_val_df['mthcaldt'], half_life_months=36
                )
            else:
                w_ft_np = np.ones(len(train_val_df), dtype=np.float32)
            w_ft = torch.tensor(w_ft_np, dtype=torch.float32, device=device)

            global_model, _, _, _ = train_structured_mdn(
                model=global_model,
                X_train_macro=X_tr_mac_ft,
                X_train_micro=X_tr_mic_ft,
                X_train_industry=X_tr_ind_ft,
                y_train=Y_tr_ft,
                w_train=w_ft,
                lr=current_lr * 0.5,
                epochs=finetune_epochs,
                patience=finetune_epochs + 1,
            )
            print(
                f"  Phase 2 fine-tune done "
                f"({finetune_epochs} epochs on Train+Val)"
            )

        duration = time.time() - start_time
        history_logs.append({
            'window': window_idx + 1,
            'train_period': f"{info['train'][0]} to {info['train'][1]}",
            'test_period': f"{info['test'][0]} to {info['test'][1]}",
            'duration_sec': duration,
            'best_val_loss': best_v_loss,
            'epochs_run': len(tr_hist),
            'train_loss_trace': tr_hist,
            'val_loss_trace': va_hist,
        })

        # 样本外预测。
        global_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma, nu = global_model(
                X_te_mac, X_te_mic, X_te_ind
            )
            pi = torch.softmax(pi_logits, dim=-1)

        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['pi_vec'] = [
            json.dumps(vec.tolist()) for vec in pi.cpu().numpy()
        ]
        preds_df['pi_logits_vec'] = [
            json.dumps(vec.tolist()) for vec in pi_logits.cpu().numpy()
        ]
        preds_df['mu_vec'] = [
            json.dumps(vec.tolist()) for vec in mu.cpu().numpy()
        ]
        preds_df['sigma_vec'] = [
            json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()
        ]

        if nu is not None:
            preds_df['nu_vec'] = [
                json.dumps(vec.tolist()) for vec in nu.cpu().numpy()
            ]
        else:
            preds_df['nu_vec'] = None

        all_oos_predictions.append(preds_df)

        if run_test == "quick" and window_idx == 1:
            break

    # 提取并保存行业 Embedding。
    embedding_weights = global_model.ind_embedding.weight.detach().cpu().numpy()
    df_emb = pd.DataFrame(
        embedding_weights,
        columns=[f'emb_dim_{i + 1}' for i in range(6)],
    )
    df_emb['ffi49'] = df_emb.index

    print()
    print(">>> Pipeline complete! Aggregating all Out-of-Sample predictions...")
    if len(all_oos_predictions) > 0:
        final_oos_df = pd.concat(all_oos_predictions, ignore_index=True)
        metrics_df = pd.DataFrame(history_logs)
        print(f"Successfully generated {len(final_oos_df)} predictions.")

In [16]:
# 指定保存路径
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

if env == 'kaggle':
    path = '/kaggle/working'
elif env == 'colab':
    path = '/content/drive/MyDrive/Colab Notebooks/'
else:
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

# 执行保存
final_oos_df.to_parquet(os.path.join(path, f'final_oos_df_{timestamp}.parquet'), index=False)
metrics_df.to_parquet(os.path.join(path, f'metrics_df_{timestamp}.parquet'), index=False)
# ES_df.to_parquet(os.path.join(path, f'ES_df_{timestamp}.parquet'), index=False)
df_emb.to_parquet(os.path.join(path, f'industry_embeddings_50x6_{timestamp}.parquet'), index=False)
print(f"保存成功！文件位置: {path}")

保存成功！文件位置: /content/drive/MyDrive/Colab Notebooks/


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
warnings.filterwarnings('ignore')
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

# 如果没有final_oos_df, 则从parquet加载
if 'final_oos_df' not in locals():
    print(">>> Loading final_oos_df from Parquet...")
    final_oos_df = pd.read_parquet('/Users/jianbinchen/Downloads/final_oos_df_20260430_0833.parquet') # local 路径
    print(f"final_oos_df loaded with shape: {final_oos_df.shape}")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

# =================================================================
# 1. 核心数学工具：混合高斯分布的 CDF 和 ES 计算
# =================================================================

def mdn_cdf(y, pi, mu, sigma, nu=None):
    """
    计算混合高斯分布在 y 处的累积概率 P(Y <= y)
    公式: F(y) = sum( pi_k * Phi((y - mu_k) / sigma_k) )
    """
    """
    兼容 N 和 t 的混合分布累积概率
    """
    if nu is None:
        # norm.cdf 是标准正态分布的累积函数
        return np.sum(pi * norm.cdf((y - mu) / sigma))
    else:
        # scipy.stats.t.cdf 的 loc(均值) 默认0，scale(标准差) 默认1，需要标准化 y
        return np.sum(pi * t.cdf((y - mu) / sigma, df=nu))


def mdn_var_objective(y, pi, mu, sigma, nu, alpha):
    """
    求解 VaR 的目标函数：F(y) - alpha = 0
    """
    return mdn_cdf(y, pi, mu, sigma, nu) - alpha

def calculate_es_analytical(alpha, pi, mu, sigma, nu=None, right_tail=False):
    """
    right_tail=False: 左尾 ES，即 E[X | X ≤ VaR_α]，用于 alpha=1%/5%/10%
    right_tail=True:  右尾 ES，即 E[X | X ≥ VaR_α]，用于 alpha=90%/95%/99%
    """
    # ==========================================
    # VaR 求根部分完全不变
    # ==========================================
    lower_bound = np.min(mu) - 10 * np.max(sigma)
    upper_bound = np.max(mu) + 10 * np.max(sigma)
    step = np.max(sigma) * 5 + 0.1

    for _ in range(100):
        val_lower = mdn_var_objective(lower_bound, pi, mu, sigma, nu, alpha)
        val_upper = mdn_var_objective(upper_bound, pi, mu, sigma, nu, alpha)
        if val_lower < 0 and val_upper > 0:
            break
        if val_lower >= 0:
            lower_bound -= step
        if val_upper <= 0:
            upper_bound += step
        step *= 1.5

    try:
        var_alpha = brentq(mdn_var_objective, lower_bound, upper_bound, args=(pi, mu, sigma, nu, alpha))
    except Exception as e:
        print(f">>> Warning: Root finding failed safely. {e}")
        return np.sum(pi * mu), np.sum(pi * mu)

    # ==========================================
    # ES 积分：根据 right_tail 切换公式
    # ==========================================
    d = (var_alpha - mu) / sigma

    if not right_tail:
        # 左尾：E[X | X ≤ VaR]，对应 alpha = 1%/5%/10%
        if nu is None:
            term1 = mu * norm.cdf(d)
            term2 = sigma * norm.pdf(d)
        else:
            term1 = mu * t.cdf(d, df=nu)
            term2 = sigma * ((nu + d**2) / (nu - 1)) * t.pdf(d, df=nu)
        es_val = -(1 / alpha) * np.sum(pi * (term1 - term2))

    else:
        # 右尾：E[X | X ≥ VaR]，对应 alpha = 90%/95%/99%
        # 公式：(1/(1-α)) * Σ pi_k * [mu_k*(1-F(d)) + sigma_k*f(d)*调整项]
        if nu is None:
            term1 = mu * (1 - norm.cdf(d))
            term2 = sigma * norm.pdf(d)
        else:
            term1 = mu * (1 - t.cdf(d, df=nu))
            term2 = sigma * ((nu + d**2) / (nu - 1)) * t.pdf(d, df=nu)
        es_val = (1 / (1 - alpha)) * np.sum(pi * (term1 + term2))

    return var_alpha, es_val

def calculate_crps_numerical(y_true, pi, mu, sigma, nu=None):
    """
    通过数值积分计算混合高斯分布的 CRPS
    """
    # 1. 构建积分网格 z
    # 因为我们的目标收益率已经用 VIX 缩放过，绝大多数值落在 [-5, 5] 之间。
    # 设置 [-10, 10] 的区间和 1000 个格点，足以保证极高的积分精度。
    z = np.linspace(-10, 10, 1000)
    
    # 2. 计算网格上每一个点的预测累积概率 F(z)
    cdf_z = np.zeros_like(z)
    for k in range(len(pi)):
        # 向量化计算：算出每一个网格点的正态 CDF 并加权
        if nu is None:
            cdf_z += pi[k] * norm.cdf((z - mu[k]) / sigma[k])
        else:
            cdf_z += pi[k] * t.cdf((z - mu[k]) / sigma[k], df=nu[k])

    # 3. 真实发生的阶跃函数 (Heaviside step function)
    # 当 z 小于真实收益率时为 0，大于等于时为 1
    step_z = (z >= y_true).astype(float)
    
    # 4. 计算平方差
    squared_diff = (cdf_z - step_z) ** 2
    
    # 5. 使用梯形法则 (Trapezoidal rule) 计算定积分
    crps_val = np.trapezoid(squared_diff, z)
    
    return crps_val

# =================================================================
# 2. 主程序：从 SQLite 读取数据并执行计算
# =================================================================
results = []

total = len(final_oos_df)
print_every = max(1, total // 50)  # 每 2% 打印一次

print(">>> Extracting Tail Risk Factors (ES & VaR)...")
# 使用 tqdm 显示进度
# for idx, row in tqdm(final_oos_df.iterrows(), total=len(final_oos_df)):
for idx, (df_idx, row) in enumerate(final_oos_df.iterrows()):
    if run_test == "quick" and idx >= 100:  # 测试模式下只跑前100行
        break
    # 每 x% 打印一次进度
    if idx % print_every == 0:
        print(f"  Progress: {idx/total*100:.0f}% ({idx}/{total})")
    # 1. 解码 JSON 字符串回 Numpy 数组
    pi = np.array(json.loads(row['pi_vec']))
    mu = np.array(json.loads(row['mu_vec']))
    sigma = np.array(json.loads(row['sigma_vec']))
    y_true = row['target_ret_final'] # 真实发生的（缩放后的）收益率

    # 🌟 2. 安全读取自由度 nu (智能判断分布类型)
    nu = None
    if 'nu_vec' in row and pd.notna(row['nu_vec']):
        nu = np.array(json.loads(row['nu_vec']))
    
    # 2. 计算风险指标：VaR 和 ES
    # 计算 1% 分位数的 VaR 和 ES
    alpha = 0.01
    var_1, es_1 = calculate_es_analytical(alpha, pi, mu, sigma, nu)
    #计算 5% 分位数的 VaR 和 ES
    alpha = 0.05
    var_5, es_5 = calculate_es_analytical(alpha, pi, mu, sigma, nu)
    # 计算 10% 分位数的 VaR 和 ES
    alpha = 0.10
    var_10, es_10 = calculate_es_analytical(alpha, pi, mu, sigma, nu)
    # 计算 90% 分位数的 VaR 和 ES
    alpha = 0.90
    var_90, es_90 = calculate_es_analytical(alpha, pi, mu, sigma, nu, right_tail=True)
    # 计算 95% 分位数的 VaR 和 ES
    alpha = 0.95
    var_95, es_95 = calculate_es_analytical(alpha, pi, mu, sigma, nu, right_tail=True)
    # 计算 99% 分位数的 VaR 和 ES
    alpha = 0.99
    var_99, es_99 = calculate_es_analytical(alpha, pi, mu, sigma, nu, right_tail=True)

    # 3. 计算 PIT (Probability Integral Transform)
    # 即：真实收益率 y_true 在预测分布中处于什么百分位
    pit_val = mdn_cdf(y_true, pi, mu, sigma, nu)

    # 新增：计算 CRPS
    crps_val = calculate_crps_numerical(y_true, pi, mu, sigma, nu)

# 🌟 4. 高阶矩计算 (方差会随 t 分布自由度变化)
    mean_pred = np.sum(pi * mu)
    if nu is None:
        component_var = sigma**2
    else:
        # t分布的真实方差公式: sigma^2 * (nu / (nu - 2))
        # (我们的网络中 nu_head 加了 2.01，所以 nu 必然大于 2，不会出现除以0的错误)
        component_var = sigma**2 * (nu / (nu - 2))
    # 混合分布方差: sum(pi * (mu^2 + sigma^2)) - mean^2
    var_pred = np.sum(pi * (mu**2 + component_var)) - mean_pred**2
    
    results.append({
        'permno': row['permno'],
        'date': row['mthcaldt'],
        'es_1': es_1,
        'var_1': var_1,
        'es_5': es_5,       
        'var_5': var_5,     
        'es_10': es_10,     
        'var_10': var_10,
        'es_90': es_90,
        'var_90': var_90,
        'es_95': es_95,
        'var_95': var_95,
        'es_99': es_99,
        'var_99': var_99,     
        'pit': pit_val,     
        'crps': crps_val,   # <--- 把 CRPS 也存进最终的 DataFrame 里
        'mean_pred': mean_pred,
        'vol_pred': np.sqrt(var_pred),
        'realized_ret': y_true
    })

ES_df = pd.DataFrame(results)

>>> Extracting Tail Risk Factors (ES & VaR)...
  Progress: 0% (0/84807)


In [18]:
ES_df.to_parquet(os.path.join(path, f'ES_df_{timestamp}.parquet'), index=False)
print(f"保存成功！文件位置: {path}")

保存成功！文件位置: /content/drive/MyDrive/Colab Notebooks/


In [21]:
if env == 'colab':
    # 运行完你的所有逻辑后，调用此命令会自动释放当前后端实例
    google.colab.runtime.unassign()